# Byte Pair Encoding

In [132]:
import tiktoken

In [133]:
tokenizer = tiktoken.get_encoding('gpt2')
print(tokenizer)

<Encoding 'gpt2'>


In [134]:
text = 'The quick brown fox jumps over the lazy dog.'
encoded_text = tokenizer.encode(
    text, 
    allowed_special={'<|endoftext|>'}
)
print(encoded_text)

[464, 2068, 7586, 21831, 18045, 625, 262, 16931, 3290, 13]


In [135]:
strings = tokenizer.decode(encoded_text)
print(strings)

The quick brown fox jumps over the lazy dog.


# Sliding Window

Input-target pairs are created by defining two variables: x and y
x contains the input tokens, and y contains the targets, which are inputs shifted by 1

In [136]:
context_size = 4
for i in range(1, context_size+1):
    x = encoded_text[:i]
    y = encoded_text[i]
    print(tokenizer.decode(x), '-->', tokenizer.decode([y]))


The -->  quick
The quick -->  brown
The quick brown -->  fox
The quick brown fox -->  jumps


# Data Loader

Stride == input size => batches do not overlap. 

More overlap increases overfitting.

In [137]:
from gpt_data_loader import create_data_loader

max_length = 4

data_loader = create_data_loader(
    text,
    batch_size=1,
    max_length=max_length,
    stride=1,
    shuffle=False
)
for inputs, targets in data_loader:
    print(tokenizer.decode(inputs[0].tolist()), '-->', tokenizer.decode(targets[0].tolist()))

 quick brown fox jumps -->  brown fox jumps over
 brown fox jumps over -->  fox jumps over the
 fox jumps over the -->  jumps over the lazy
 jumps over the lazy -->  over the lazy dog
 over the lazy dog -->  the lazy dog.


# Token Embeddings

In [138]:
import torch

vocab_size = 6
output_dim = 3 # embedding size
torch.manual_seed(101)

embedding_layer = torch.nn.Embedding(
    vocab_size,
    output_dim
) # optimized during the training

print(embedding_layer.weight, end='\n\n')
token_idx = torch.tensor([1])
print(embedding_layer(token_idx))

Parameter containing:
tensor([[-0.6028, -0.1208,  1.0552],
        [ 0.7783,  1.0674,  0.1143],
        [-0.5580,  0.3066, -1.0023],
        [-0.5726,  0.9192,  0.5596],
        [ 1.7718, -0.9621,  0.0615],
        [-0.9060,  0.8110,  1.0818]], requires_grad=True)

tensor([[0.7783, 1.0674, 0.1143]], grad_fn=<EmbeddingBackward0>)


# Position

Self-attention doesn't have a notion of position or order for the tokens within a sequence.

For each position in the input sequence, a unique vector is added to the token's embedding to convey its exact location.

Positional embeddings can be absolute or relative (distance between tokens).

GPT uses absolute embeddings that are optimized during the training.

In [139]:
vocab_size = 50257
output_dim = 256
print(inputs.shape)
real_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
token_embeddings = real_embedding_layer(inputs)
print(token_embeddings.shape)
print()

positional_layer = torch.nn.Embedding(max_length, output_dim)

# Pobranie kolejnych wierszy z macierzy wag.
# Równoważne z positional_layer.weight
positional_embeddings = positional_layer(
    torch.arange(context_size)
)
print(positional_embeddings.shape)

input_embedings = token_embeddings + positional_embeddings
print(input_embedings.shape)
print(torch.arange(context_size))
print(input_embedings.sum())

torch.Size([1, 4])
torch.Size([1, 4, 256])

torch.Size([4, 256])
torch.Size([1, 4, 256])
tensor([0, 1, 2, 3])
tensor(18.3668, grad_fn=<SumBackward0>)
